In [52]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/fasgadhsxnzmjj/mmd-dataset/centralVN_dataWeather.csv
/kaggle/input/datasets/fasgadhsxnzmjj/mmd-dataset/aqi_centralVN_daily.csv
/kaggle/input/datasets/fasgadhsxnzmjj/mmd-dataset/aqi_southVN_daily.csv
/kaggle/input/datasets/fasgadhsxnzmjj/mmd-dataset/northVN_dataThoiTiet.csv
/kaggle/input/datasets/fasgadhsxnzmjj/mmd-dataset/southVN_dataAIR.csv
/kaggle/input/datasets/fasgadhsxnzmjj/mmd-dataset/centralVN.csv
/kaggle/input/datasets/fasgadhsxnzmjj/mmd-dataset/southVN_dataWeather.csv
/kaggle/input/datasets/fasgadhsxnzmjj/mmd-dataset/centralVN_dataAIR.csv
/kaggle/input/datasets/fasgadhsxnzmjj/mmd-dataset/northVN_dataAIR.csv
/kaggle/input/datasets/fasgadhsxnzmjj/mmd-dataset/southVN.csv
/kaggle/input/datasets/fasgadhsxnzmjj/mmd-dataset/northVN.csv
/kaggle/input/datasets/fasgadhsxnzmjj/mmd-dataset/aqi_northVN_daily.csv


In [53]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

import tensorflow as tf
from keras.models import Sequential
from keras.layers import LSTM, Dropout, Dense, Input

import matplotlib.pyplot as plt

# 1.Load data

In [54]:
df = pd.read_csv('/kaggle/input/datasets/fasgadhsxnzmjj/mmd-dataset/northVN.csv')

In [55]:
df = df[df['city'] == 'Ha Noi']

In [56]:
df.head()

,district,city,time,temperature_2m (°C),relative_humidity_2m (%),apparent_temperature (°C),precipitation (mm),rain (mm),cloud_cover (%),cloud_cover_low (%),...,pm2_5 (μg/m³),carbon_monoxide (μg/m³),nitrogen_dioxide (μg/m³),sulphur_dioxide (μg/m³),ozone (μg/m³),aerosol_optical_depth (),dust (μg/m³),uv_index (),uv_index_clear_sky (),aqi_h
0,Ba Dinh,Ha Noi,2022-08-04 07:00:00,28.5,87,35.2,0.0,0.0,8,0,...,40.3,595.0,29.7,16.8,24.0,0.43,0.0,0.75,0.80,81
1,Ba Dinh,Ha Noi,2022-08-04 08:00:00,29.5,82,36.6,0.0,0.0,100,0,...,30.0,552.0,25.0,18.2,49.0,0.52,0.0,2.10,2.35,72
2,Ba Dinh,Ha Noi,2022-08-04 09:00:00,30.4,80,38.0,0.0,0.0,100,5,...,32.7,492.0,18.4,20.2,84.0,0.59,0.0,3.95,4.55,70
3,Ba Dinh,Ha Noi,2022-08-04 10:00:00,31.8,73,39.4,0.0,0.0,94,23,...,34.9,429.0,11.2,22.0,128.0,0.68,0.0,4.70,6.75,70
4,Ba Dinh,Ha Noi,2022-08-04 11:00:00,32.5,71,41.1,0.0,0.0,92,3,...,38.3,414.0,8.6,21.5,154.0,0.75,0.0,5.10,8.25,71


# 2.Data cleaning

In [57]:
print(df.columns)

Index(['district', 'city', 'time', 'temperature_2m (°C)',
       'relative_humidity_2m (%)', 'apparent_temperature (°C)',
       'precipitation (mm)', 'rain (mm)', 'cloud_cover (%)',
       'cloud_cover_low (%)', 'cloud_cover_mid (%)', 'cloud_cover_high (%)',
       'wind_speed_10m (km/h)', 'wind_speed_100m (km/h)',
       'wind_direction_10m (°)', 'soil_temperature_0_to_7cm (°C)',
       'soil_temperature_7_to_28cm (°C)', 'soil_temperature_28_to_100cm (°C)',
       'soil_temperature_100_to_255cm (°C)', 'soil_moisture_0_to_7cm (m³/m³)',
       'soil_moisture_7_to_28cm (m³/m³)', 'soil_moisture_28_to_100cm (m³/m³)',
       'soil_moisture_100_to_255cm (m³/m³)', 'pm10 (μg/m³)', 'pm2_5 (μg/m³)',
       'carbon_monoxide (μg/m³)', 'nitrogen_dioxide (μg/m³)',
       'sulphur_dioxide (μg/m³)', 'ozone (μg/m³)', 'aerosol_optical_depth ()',
       'dust (μg/m³)', 'uv_index ()', 'uv_index_clear_sky ()', 'aqi_h'],
      dtype='object')


In [58]:
# chọn ra những cột quan trọng
selected_columns = [
    'district', 'city', 'time',

    # pollutants
    'pm2_5 (μg/m³)', 'pm10 (μg/m³)',
    'carbon_monoxide (μg/m³)',
    'nitrogen_dioxide (μg/m³)',
    'sulphur_dioxide (μg/m³)',
    'ozone (μg/m³)',
    

    'aqi_h'
]

df = df[selected_columns]

print(df.shape)

(919950, 10)


In [59]:
df['time'] = pd.to_datetime(df['time'])

In [60]:
df.isnull().sum()

district                    0
city                        0
time                        0
pm2_5 (μg/m³)               0
pm10 (μg/m³)                0
carbon_monoxide (μg/m³)     0
nitrogen_dioxide (μg/m³)    0
sulphur_dioxide (μg/m³)     0
ozone (μg/m³)               0
aqi_h                       0
dtype: int64

In [61]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 919950 entries, 0 to 919949
Data columns (total 10 columns):
 #   Column                    Non-Null Count   Dtype         
---  ------                    --------------   -----         
 0   district                  919950 non-null  object        
 1   city                      919950 non-null  object        
 2   time                      919950 non-null  datetime64[ns]
 3   pm2_5 (μg/m³)             919950 non-null  float64       
 4   pm10 (μg/m³)              919950 non-null  float64       
 5   carbon_monoxide (μg/m³)   919950 non-null  float64       
 6   nitrogen_dioxide (μg/m³)  919950 non-null  float64       
 7   sulphur_dioxide (μg/m³)   919950 non-null  float64       
 8   ozone (μg/m³)             919950 non-null  float64       
 9   aqi_h                     919950 non-null  int64         
dtypes: datetime64[ns](1), float64(6), int64(1), object(2)
memory usage: 77.2+ MB


In [62]:
df = df.sort_values(['city', 'district', 'time']).reset_index(drop=True)
df.head()

,district,city,time,pm2_5 (μg/m³),pm10 (μg/m³),carbon_monoxide (μg/m³),nitrogen_dioxide (μg/m³),sulphur_dioxide (μg/m³),ozone (μg/m³),aqi_h
0,Ba Dinh,Ha Noi,2022-08-04 07:00:00,40.3,58.0,595.0,29.7,16.8,24.0,81
1,Ba Dinh,Ha Noi,2022-08-04 08:00:00,30.0,43.5,552.0,25.0,18.2,49.0,72
2,Ba Dinh,Ha Noi,2022-08-04 09:00:00,32.7,47.3,492.0,18.4,20.2,84.0,70
3,Ba Dinh,Ha Noi,2022-08-04 10:00:00,34.9,50.3,429.0,11.2,22.0,128.0,70
4,Ba Dinh,Ha Noi,2022-08-04 11:00:00,38.3,55.2,414.0,8.6,21.5,154.0,71


# 3. Model

## Chọn feature

In [63]:
features = [
    'pm2_5 (μg/m³)', 'pm10 (μg/m³)',
    'carbon_monoxide (μg/m³)',
    'nitrogen_dioxide (μg/m³)',
    'sulphur_dioxide (μg/m³)',
    'ozone (μg/m³)'

    
]

target = 'aqi_h'

## Train test split

In [64]:
split = int(len(df) * 0.8)

train = df.iloc[:split].copy()
test = df.iloc[split:].copy()

## Scale data

In [65]:
from sklearn.preprocessing import StandardScaler

scaler_x = StandardScaler()
scaler_y = StandardScaler()

train[features] = scaler_x.fit_transform(
    train[features]
)

test[features] = scaler_x.transform(
    test[features]
)

train[[target]] = scaler_y.fit_transform(
    train[[target]]
)

test[[target]] = scaler_y.transform(
    test[[target]]
)

## Tạo SEQUENCE (MULTI-STEP)

In [66]:
# SỬA LẠI Ô SỐ 16:
def create_sequences_multi(data, features, target, window=24, horizon=24):
    x = []
    y = []
    
    # Lấy mảng giá trị tách biệt rõ ràng
    x_values = data[features].values
    y_values = data[target].values

    for i in range(len(data) - window - horizon):
        x.append(x_values[i : i + window])
        y.append(y_values[i + window : i + window + horizon])

    return np.array(x), np.array(y)

# Chạy lại lệnh tạo dữ liệu
x_train, y_train = create_sequences_multi(train, features, target)
x_test, y_test = create_sequences_multi(test, features, target)

print("x_train:", x_train.shape) # Sẽ có dạng (24484, 24, 7) do bớt 1 cột aqi
print("y_train:", y_train.shape)

x_train: (735912, 24, 6)
y_train: (735912, 24)


## Buid model LSTM

In [67]:
from keras.models import Sequential
from keras.layers import (
    LSTM,
    Dense,
    Dropout,
    Input
)

model = Sequential([

    Input(
        shape=(24, len(features))
    ),

    LSTM(
        64,
        return_sequences=True
    ),

    Dropout(0.2),

    LSTM(32),

    Dropout(0.2),

    Dense(
        32,
        activation='relu'
    ),

    Dense(24)

])

## Compile Model

In [68]:
model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

## EarlyStopping

In [69]:
from keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

## Train

In [70]:
history = model.fit(
    x_train,
    y_train,

    validation_data=(
        x_test,
        y_test
    ),

    epochs=50,

    batch_size=128,

    callbacks=[
        early_stop
    ]
)

Epoch 1/50
5750/5750 ━━━━━━━━━━━━━━━━━━━━ 47s 8ms/step - loss: 0.2845 - mae: 0.3759 - val_loss: 0.2071 - val_mae: 0.2928
Epoch 2/50
5750/5750 ━━━━━━━━━━━━━━━━━━━━ 44s 8ms/step - loss: 0.1850 - mae: 0.2963 - val_loss: 0.1843 - val_mae: 0.2814
Epoch 3/50
5750/5750 ━━━━━━━━━━━━━━━━━━━━ 44s 8ms/step - loss: 0.1527 - mae: 0.2699 - val_loss: 0.1789 - val_mae: 0.2914
Epoch 4/50
5750/5750 ━━━━━━━━━━━━━━━━━━━━ 44s 8ms/step - loss: 0.1341 - mae: 0.2541 - val_loss: 0.1697 - val_mae: 0.2893
Epoch 5/50
5750/5750 ━━━━━━━━━━━━━━━━━━━━ 44s 8ms/step - loss: 0.1224 - mae: 0.2436 - val_loss: 0.1671 - val_mae: 0.2897
Epoch 6/50
5750/5750 ━━━━━━━━━━━━━━━━━━━━ 44s 8ms/step - loss: 0.1135 - mae: 0.2355 - val_loss: 0.1600 - val_mae: 0.2849
Epoch 7/50
5750/5750 ━━━━━━━━━━━━━━━━━━━━ 44s 8ms/step - loss: 0.1070 - mae: 0.2288 - val_loss: 0.1570 - val_mae: 0.2845
Epoch 8/50
5750/5750 ━━━━━━━━━━━━━━━━━━━━ 44s 8ms/step - loss: 0.1019 - mae: 0.2235 - val_loss: 0.1680 - val_mae: 0.2982
Epoch 9/50
5750/5750 ━━━━━━━━━━━

## Dự đoán

In [71]:
pred = model.predict(x_test)

5749/5749 ━━━━━━━━━━━━━━━━━━━━ 12s 2ms/step


## Inverse Scale

In [72]:
y_test_inv = scaler_y.inverse_transform(
    y_test.reshape(-1,1)
).reshape(
    y_test.shape
)

pred_inv = scaler_y.inverse_transform(
    pred.reshape(-1,1)
).reshape(
    pred.shape
)

## Evaluate

In [73]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

mae = mean_absolute_error(
    y_test_inv.flatten(),
    pred_inv.flatten()
)

mse = mean_squared_error(
    y_test_inv.flatten(),
    pred_inv.flatten()
)

rmse = np.sqrt(mse)

r2 = r2_score(
    y_test_inv.flatten(),
    pred_inv.flatten()
)

print("MAE :", round(mae,4))
print("MSE :", round(mse,4))
print("RMSE:", round(rmse,4))
print("R2  :", round(r2,4))

MAE : 11.691
MSE : 265.1601
RMSE: 16.2837
R2  : 0.8486


In [74]:
# Lưu toàn bộ mô hình thành một file duy nhất
model.save('aqi_lstm_model_HN.keras')

In [75]:
import joblib

# Lưu bộ chuẩn hóa đầu vào và đầu ra
joblib.dump(scaler_x, 'scaler_x.joblib')
joblib.dump(scaler_y, 'scaler_y.joblib')

['scaler_y.joblib']